# DevRev Search Dataset

Loading and exploring the `devrev/search` dataset from Hugging Face.

This copy replaces the original FAISS-only retrieval step with the v2 search stack: BM25 + dense retrieval + RRF fusion + cross-encoder reranking.


In [1]:
from datasets import load_dataset
import pandas as pd

/home/adhi/projects/devrev-search-bench/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Annotated Queries
Queries paired with annotated (golden) article chunks for training/validation.

In [2]:
# Load annotated queries
annotated_queries = load_dataset("devrev/search", "annotated_queries")
print(annotated_queries)

DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'retrievals'],
        num_rows: 291
    })
})


In [3]:
# Convert to DataFrame and display
annotated_df = annotated_queries["train"].to_pandas()
annotated_df.head()

,query_id,query,retrievals
0,0ae94217-c6a0-4895-83a2-841a95f01637,create DevRev ticket from Microsoft Teams,"[{'id': 'ART-4216_KNOWLEDGE_NODE-26', 'text': ..."
1,d0b209b3-6cea-46d8-bfac-bd0e286ea21b,workflow builder auto close ticket after 48 ho...,"[{'id': 'ART-2012_KNOWLEDGE_NODE-24', 'text': ..."
2,40c1aa6f-cd21-46ab-8f6f-76fdc267b584,automated reminder to customer ticket will be ...,"[{'id': 'ART-3068_KNOWLEDGE_NODE-24', 'text': ..."
3,e47d883f-b712-4f98-bd06-14ade143e3c2,connect Bitbucket account to DevRev account,"[{'id': 'ART-2030_KNOWLEDGE_NODE-27', 'text': ..."
4,2e6f9413-15ac-4974-a380-7aa22fc98a61,use of workflows in DevRev,"[{'id': 'ART-1961_KNOWLEDGE_NODE-28', 'text': ..."


In [4]:
# Sample a single annotated query example
annotated_queries["train"][0]

{'query_id': '0ae94217-c6a0-4895-83a2-841a95f01637',
 'query': 'create DevRev ticket from Microsoft Teams',
 'retrievals': [{'id': 'ART-4216_KNOWLEDGE_NODE-26',
   'text': 'DevRev Object | Sync to DevRev |\\n| --- | --- | --- |\\n| Plan | Parts | \\xe2\\x9c\\x85 |\\n| User | Identity/DevUser | \\xe2\\x9c\\x85 |\\n| Channel | Chat | \\xe2\\x9c\\x85 |\\n| Attachments in Message/Thread/Task | Artifacts on Article | \\xe2\\x9c\\x85 |\\n| Message | Comment | \\xe2\\x9c\\x85 |\\n| Thread | Comment | \\xe2\\x9c\\x85 |\\n| Task | Issue/Ticket | \\xe2\\x9c\\x85 |\\n\\nImporting from Microsoft Teams\\n------------------------------\\n\\nFollow the steps below to import from Microsoft Teams:\\n\\n1. Go to',
   'title': 'Microsoft Teams AirSync | AirSync | Snap-ins | DevRev'},
  {'id': 'ART-4216_KNOWLEDGE_NODE-29',
   'text': 'with many\\nattachments. DevRev honors the Microsoft Graph API rate limits and back-off and resumes automatically.\\n\\nPost import options\\n-------------------\\n\\nAfter 

## 2. Load Test Queries
Held-out queries used for evaluation.

In [5]:
# Load test queries
test_queries = load_dataset("devrev/search", "test_queries")
print(test_queries)

DatasetDict({
    test: Dataset({
        features: ['query_id', 'query'],
        num_rows: 92
    })
})


In [6]:
# Convert to DataFrame and display
test_df = test_queries["test"].to_pandas()
test_df.head()

,query_id,query
0,a97f93d2-410a-431f-ae9a-1e23ed35d74c,end customer organization name not appearing i...
1,7dd7e2b4-9349-4535-8007-1d706e0fabff,Android SDK session generated with Unknown user
2,4bc92187-cdaa-4c20-b189-abd1672e5a71,email reply received on wrong ticket
3,4d9878e8-f746-4df5-8bf6-f9444989b385,manage access and privileges in DevRev
4,483151ec-aff4-4569-b3df-651f578b61d8,SSO setup SAML IDP metadata connection string ...


In [7]:
# Sample a single test query example
test_queries["test"][0]

{'query_id': 'a97f93d2-410a-431f-ae9a-1e23ed35d74c',
 'query': 'end customer organization name not appearing in ticket or conversation'}

## 3. Load Knowledge Base
Article chunks from DevRev's customer-facing support documentation.

In [8]:
# Load knowledge base
knowledge_base = load_dataset("devrev/search", "knowledge_base")
print(knowledge_base)

DatasetDict({
    corpus: Dataset({
        features: ['id', 'text', 'title'],
        num_rows: 65224
    })
})


In [9]:
# Convert to DataFrame and display
knowledge_df = knowledge_base["corpus"].to_pandas()
knowledge_df.head()

,id,text,title
0,ART-17711_KNOWLEDGE_NODE-0,b'We ran into a case where an AirSync was star...,Sync fails when original sync owners loses per...
1,ART-17711_KNOWLEDGE_NODE-1,access.\n\nOnce Person A was re-added with the...,Sync fails when original sync owners loses per...
2,ART-17650_KNOWLEDGE_NODE-0,"b""American cybersecurity leader unifies securi...",American cybersecurity leader unifies security...
3,ART-17650_KNOWLEDGE_NODE-1,DevRev\n======================================...,American cybersecurity leader unifies security...
4,ART-17650_KNOWLEDGE_NODE-2,solutions help organisations build and deploy ...,American cybersecurity leader unifies security...


In [10]:
# Sample a single knowledge base chunk
knowledge_base["corpus"][0]

{'id': 'ART-17711_KNOWLEDGE_NODE-0',
 'text': "b'We ran into a case where an AirSync was started by one person (Person A) and later failed. Another user (Person B) tried to click Retry, but it didn\\xe2\\x80\\x99t work. The logs showed 401 and 403 errors in communication between the snap-in and the snap-in manager.\\n\\nIt turned out that AirSync assigns the sync owner to whoever started it. Since Person A had been removed from the org or lost permissions, the retry failed \\xe2\\x80\\x94 the system still expected the original owner to have valid",
 'title': 'Sync fails when original sync owners loses permissions'}

## 4. Dataset Summary

In [11]:
print("=" * 60)
print("DevRev Search Dataset Summary")
print("=" * 60)
print(f"\nAnnotated Queries:")
print(annotated_queries)
print(f"\nTest Queries:")
print(test_queries)
print(f"\nKnowledge Base:")
print(knowledge_base)
print("\n" + "=" * 60)

DevRev Search Dataset Summary

Annotated Queries:
DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'retrievals'],
        num_rows: 291
    })
})

Test Queries:
DatasetDict({
    test: Dataset({
        features: ['query_id', 'query'],
        num_rows: 92
    })
})

Knowledge Base:
DatasetDict({
    corpus: Dataset({
        features: ['id', 'text', 'title'],
        num_rows: 65224
    })
})



---
## 5. Index Knowledge Base with FAISS

Use either OpenAI (`text-embedding-3-small`) or Ollama (`qwen3-embedding:0.6b`) embeddings with the same FAISS pipeline.

## 5. Build the v2 Retrieval Pipeline

This version swaps the original FAISS-only search for the v2 retrieval pipeline used in `devrev_search_v2.ipynb`.


In [12]:
import os
import re
import json
import math
import pickle

import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from tqdm.auto import tqdm
from typing import Callable

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

EMBED_PATH = "bge_embeddings.npy"
INDEX_PATH = "bge_faiss.index"
STATE_PATH = "corpus_state_v2.pkl"
OUTPUT_JSON = "test_queries_results_v2.json"
OUTPUT_PARQUET = "test_queries_results_v2.parquet"

print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"Reranker model: {RERANKER_MODEL_NAME}")


Embedding model: BAAI/bge-base-en-v1.5
Reranker model: BAAI/bge-reranker-v2-m3


In [13]:
# Build corpus structures for retrieval
corpus = knowledge_base["corpus"]

kb_id_to_text = {}
kb_id_to_title = {}
corpus_ids = []
corpus_texts = []

for item in tqdm(corpus, desc="Building corpus index"):
    doc_id = item["id"]
    title = (item.get("title", "") or "").strip()
    text = (item.get("text", "") or "").strip()
    combined = f"{title}. {text}".strip(". ")

    kb_id_to_text[doc_id] = text
    kb_id_to_title[doc_id] = title
    corpus_ids.append(doc_id)
    corpus_texts.append(combined)

print(f"Corpus size: {len(corpus_ids):,} documents")


Building corpus index: 100%|██████████| 65224/65224 [00:01<00:00, 46582.82it/s]

Corpus size: 65,224 documents


## 6. Stage 1: BM25 Lexical Retrieval


In [14]:
STOP_WORDS = {
    "the", "a", "an", "is", "in", "of", "to", "and", "or", "for", "on", "at", "by",
    "with", "from", "be", "are", "was", "were", "this", "that", "it", "as", "not",
    "can", "do", "does", "have", "has", "had", "will", "would", "how", "what",
    "when", "where", "which", "who", "why", "if", "then", "so", "but"
}


def tokenize(text: str) -> list[str]:
    tokens = re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()
    return [token for token in tokens if token not in STOP_WORDS and len(token) > 1]


print("Tokenizing corpus for BM25...")
tokenized_corpus = [tokenize(text) for text in tqdm(corpus_texts, desc="Tokenizing")]

print("Building BM25 index...")
bm25 = BM25Okapi(tokenized_corpus, k1=1.6, b=0.75)
print("BM25 index ready")


def bm25_retrieve(query: str, top_k: int = 100) -> list[tuple[str, float]]:
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(corpus_ids[i], float(scores[i])) for i in top_indices]


Tokenizing corpus for BM25...


Tokenizing: 100%|██████████| 65224/65224 [00:02<00:00, 26545.20it/s]


Building BM25 index...
BM25 index ready


## 7. Stage 2: Dense Retrieval with FAISS


In [15]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
print(f"Loaded embedding model: {EMBED_MODEL_NAME}")

if os.path.exists(EMBED_PATH):
    print("Loading cached dense embeddings...")
    doc_embeddings = np.load(EMBED_PATH)
else:
    print(f"Encoding {len(corpus_texts):,} documents...")
    doc_embeddings = embed_model.encode(
        corpus_texts,
        batch_size=256,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(EMBED_PATH, doc_embeddings)
    print(f"Saved embeddings to {EMBED_PATH}")

embedding_dim = int(doc_embeddings.shape[1])
print(f"Embedding matrix shape: {doc_embeddings.shape}")

if os.path.exists(INDEX_PATH):
    print("Loading cached FAISS index...")
    faiss_index = faiss.read_index(INDEX_PATH)
else:
    print("Building FAISS IVF index...")
    nlist = 256
    quantizer = faiss.IndexFlatIP(embedding_dim)
    faiss_index = faiss.IndexIVFFlat(quantizer, embedding_dim, nlist, faiss.METRIC_INNER_PRODUCT)
    faiss_index.train(doc_embeddings.astype(np.float32))
    faiss_index.add(doc_embeddings.astype(np.float32))
    faiss.write_index(faiss_index, INDEX_PATH)
    print(f"Saved FAISS index to {INDEX_PATH}")

faiss_index.nprobe = 32
print(f"Index contains {faiss_index.ntotal:,} vectors")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1338.46it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded embedding model: BAAI/bge-base-en-v1.5
Encoding 65,224 documents...


Batches: 100%|██████████| 255/255 [22:08<00:00,  5.21s/it]


Saved embeddings to bge_embeddings.npy
Embedding matrix shape: (65224, 768)
Building FAISS IVF index...
Saved FAISS index to bge_faiss.index
Index contains 65,224 vectors


In [16]:
def dense_retrieve(query: str, top_k: int = 100) -> list[tuple[str, float]]:
    query_embedding = embed_model.encode(
        BGE_QUERY_PREFIX + query,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).reshape(1, -1).astype(np.float32)

    scores, indices = faiss_index.search(query_embedding, top_k)
    return [
        (corpus_ids[idx], float(score))
        for idx, score in zip(indices[0], scores[0])
        if idx >= 0
    ]


## 8. Hybrid Fusion and Reranking


In [17]:
def reciprocal_rank_fusion(
    ranked_lists: list[list[tuple[str, float]]],
    k: int = 60,
    top_k: int = 100,
) -> list[tuple[str, float]]:
    rrf_scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, (doc_id, _) in enumerate(ranked):
            rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(rrf_scores.items(), key=lambda item: item[1], reverse=True)[:top_k]


def hybrid_retrieve(query: str, bm25_k: int = 100, dense_k: int = 100, top_k: int = 100):
    bm25_results = bm25_retrieve(query, top_k=bm25_k)
    dense_results = dense_retrieve(query, top_k=dense_k)
    return reciprocal_rank_fusion([bm25_results, dense_results], top_k=top_k)


reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512)
print(f"Loaded reranker: {RERANKER_MODEL_NAME}")


def rerank(query: str, candidates: list[tuple[str, float]], top_k: int = 10) -> list[tuple[str, float]]:
    if not candidates:
        return []

    pairs = []
    valid_ids = []
    for doc_id, _ in candidates:
        text = kb_id_to_text.get(doc_id, "")
        if text:
            pairs.append([query, text[:512]])
            valid_ids.append(doc_id)

    if not pairs:
        return candidates[:top_k]

    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = sorted(zip(valid_ids, scores.tolist()), key=lambda item: item[1], reverse=True)
    return ranked[:top_k]


def full_pipeline(
    query: str,
    bm25_k: int = 100,
    dense_k: int = 100,
    rerank_k: int = 50,
    final_k: int = 10,
) -> list[tuple[str, float]]:
    candidates = hybrid_retrieve(query, bm25_k=bm25_k, dense_k=dense_k, top_k=rerank_k)
    return rerank(query, candidates, top_k=final_k)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1502.39it/s]


Loaded reranker: BAAI/bge-reranker-v2-m3


## 9. Search the Knowledge Base


In [18]:
def search(query: str, top_k: int = 5):
    """Search the knowledge base with the v2 hybrid retrieval pipeline."""
    results = full_pipeline(query, final_k=top_k)

    formatted = []
    for rank, (doc_id, score) in enumerate(results, start=1):
        formatted.append({
            "rank": rank,
            "score": float(score),
            "id": doc_id,
            "title": kb_id_to_title.get(doc_id, ""),
            "text": kb_id_to_text.get(doc_id, ""),
        })
    return formatted


In [19]:
# Test search with a sample query
query = "How do I set up AirSync?"
results = search(query, top_k=5)

print(f"Query: {query}")
print("=" * 60)
for result in results:
    print(f"[Rank {result['rank']}] Score: {result['score']:.4f}")
    print(f"Doc ID: {result['id']}")
    print(f"Title: {result['title']}")
    print(f"Text: {result['text'][:300]}...")
    print("-" * 40)


Query: How do I set up AirSync?
[Rank 1] Score: 0.9968
Doc ID: ART-2045_KNOWLEDGE_NODE-29
Title: AirSync | Snap-ins | DevRev
Text: Setting up a new AirSync\n\n![]()\n\nFor best results, AirSyncs should be done using an administrator account on the external source. This ensures all necessary permissions are available to complete the import successfully.\n\nWhether you want to perform only a one-time import or set up an ongoing s...
----------------------------------------
[Rank 2] Score: 0.9892
Doc ID: ART-2044_KNOWLEDGE_NODE-34
Title: ServiceNow AirSync | AirSync | Snap-ins | DevRev
Text: a regular basis. By default, the sync occurs once an hour.\n\nTo configure periodic sync, follow these steps:\n\n1. Go to **Settings** > **Integrations** > **AirSyncs**.\n2. Locate the previously imported project.\n3. Select the **\xe2\x8b\xae** > **Set Periodic Sync** option.\n\nThe **Enable automa...
----------------------------------------
[Rank 3] Score: 0.9786
Doc ID: ART-4215_KNOWLEDGE_NODE-27
T

## 10. Generate Test Query Predictions


In [20]:
TOP_K = 10

test_results = []

for item in tqdm(test_queries["test"], desc="Processing test queries"):
    query_id = item["query_id"]
    query = item["query"]

    retrieved = full_pipeline(query, bm25_k=100, dense_k=100, rerank_k=50, final_k=TOP_K)
    retrievals = [
        {
            "id": doc_id,
            "text": kb_id_to_text.get(doc_id, ""),
            "title": kb_id_to_title.get(doc_id, ""),
        }
        for doc_id, _ in retrieved
    ]

    test_results.append({
        "query_id": query_id,
        "query": query,
        "retrievals": retrievals,
    })

print(f"Processed {len(test_results)} test queries")


Processing test queries: 100%|██████████| 92/92 [11:39<00:00,  7.61s/it]

Processed 92 test queries


In [21]:
print("Sample result:")
print(json.dumps(test_results[0], indent=2, default=str)[:1500])


Sample result:
{
  "query_id": "a97f93d2-410a-431f-ae9a-1e23ed35d74c",
  "query": "end customer organization name not appearing in ticket or conversation",
  "retrievals": [
    {
      "id": "ART-1953_KNOWLEDGE_NODE-29",
      "text": "[support@yourdomain.com](mailto:support@yourdomain.com)\\n* **Subject**: \"You are missing messages from <Org Name>\"\\n\\nReply to the customer on a ticket\\n---------------------------------\\n\\n* **Trigger**: When a reply is made to a customer on a ticket.\\n* **Action**: The system sends out a notification to the customer with the reply message.\\n* **Sender**: {Company\\\\_Name} [support@yourdomain.com](mailto:support@yourdomain.com)\\n* **Subject**: \"[{Company\\\\_Name}] Update on TKT-XXX\"\\n\\nTicket",
      "title": "Customer email notifications | Computer by DevRev | DevRev"
    },
    {
      "id": "ART-1981_KNOWLEDGE_NODE-30",
      "text": "conversation of which you are not the owner, let the owner know to respond. It's beneficial to reta

In [22]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as file:
    json.dump(test_results, file, indent=2)

results_df = pd.DataFrame(test_results)
results_df.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Saved JSON results to {OUTPUT_JSON}")
print(f"Saved parquet results to {OUTPUT_PARQUET}")
print(f"Queries: {len(test_results)}")
print(f"Retrievals per query: {TOP_K}")


Saved JSON results to test_queries_results_v2.json
Saved parquet results to test_queries_results_v2.parquet
Queries: 92
Retrievals per query: 10


## 11. Save and Reload Retrieval State


In [23]:
state = {
    "corpus_ids": corpus_ids,
    "kb_id_to_text": kb_id_to_text,
    "kb_id_to_title": kb_id_to_title,
}

with open(STATE_PATH, "wb") as file:
    pickle.dump(state, file)

print(f"Saved retrieval state to {STATE_PATH}")
print(f"Dense index cache: {INDEX_PATH}")
print(f"Dense embedding cache: {EMBED_PATH}")


Saved retrieval state to corpus_state_v2.pkl
Dense index cache: bge_faiss.index
Dense embedding cache: bge_embeddings.npy


## 12. Load Saved State (Optional)
Use this to reload the cached retrieval components without rebuilding the corpus structures.


In [24]:
# with open(STATE_PATH, "rb") as file:
#     state = pickle.load(file)
#
# corpus_ids = state["corpus_ids"]
# kb_id_to_text = state["kb_id_to_text"]
# kb_id_to_title = state["kb_id_to_title"]
#
# doc_embeddings = np.load(EMBED_PATH)
# faiss_index = faiss.read_index(INDEX_PATH)
# faiss_index.nprobe = 32
#
# embed_model = SentenceTransformer(EMBED_MODEL_NAME)
# reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512)
# print("Reloaded retrieval state")
